# OpenVLA transfer arm — Colab T4 probe

Runs the **robot-pretrained OpenVLA-7B transfer arm** in 4-bit on a **free Colab T4**.
The 50-step smoke (2026-07-16) already proved it loads and trains without OOM/dtype errors.
This is now a **600-step probe (~50 min)** answering the question the smoke could not:
**does the pipeline learn anything from the camera at all**, before you book the lab GPU?

**Scope / what this is NOT.** Still not a *finished* result. Cells 7–9 now go past the probe:
they **score it with the offline eval** and run the **scratch control arm** to ask whether the
learning is *robot pretraining* or just finetuning on drone data. The scratch base is a full
~14 GB fp16 backbone (not 4-bit), so it is **borderline on a free 16 GB T4** — it may OOM, in
which case use Colab Pro (an L4 has 24 GB) or the lab GPU. The **prismatic** gold-standard
control needs 24 GB + the `prismatic-vlms` package and stays lab-only. The clean, matched-
precision 3-arm ablation still belongs on the lab GPU with `configs/openvla.yaml`.

**How to read the loss.** The trainer prints a **marginal-only loss floor** (2.563 nats for
waypoint) — what a model scores knowing the action distribution but ignoring the image.
Breaking below it means the model is using the camera; settling at it means it is not.
The 50-step runs both sat ~0.1 nats off their floor, which is all 200 samples buys.

`velocity` mode is ill-posed from a single frame (motion is invisible in one image), so every
arm converges to the marginal and the ablation returns a null result by construction. That is
why this config uses `waypoint`. **Do not switch it back.**

### Before you start
1. **Runtime -> Change runtime type -> T4 GPU.**
2. **Push the repo first, then re-run cell 2 AND cell 4 on every retry.** This notebook clones
   `main`, so a fix must be pushed *before* the clone. Re-running only the train cell in a live
   session silently reuses the OLD clone — that is how the 15:18 run got new data but stale
   logging code. And cell 2 wipes the repo dir, so **always follow it with cell 4** to re-link
   the data — cheap now, since the data lives outside the repo and cell 4 becomes a no-op
   symlink after the first extract.
3. **Upload the data to Google Drive.** `data/uzh_fpv/` is git-ignored, so it travels via
   Drive. It must be data converted at or after commit `6789b8f` — earlier conversions
   clipped velocity at 5 m/s (saturating a 13 m/s dataset) and carry no `poses.npy`, which
   waypoint mode requires. Rebuild it on your laptop with:
   ```bash
   python scripts/convert_uzh_fpv.py --src data/raw/uzh_fpv --dst data/uzh_fpv
   tar czf uzh_fpv.tar.gz -C data uzh_fpv
   ```
   then upload `uzh_fpv.tar.gz` to Drive at `MyDrive/alpamayo/uzh_fpv.tar.gz`, **deleting
   the old copy first** (a stale one is indistinguishable by name and silently reproduces
   the saturated run). Adjust the path in the data cell if you put it elsewhere.

## 1. Confirm the GPU is a T4
Turing (T4) / Pascal (P100) have no bf16 tensor cores — that's why this run uses fp16.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Clone the repo

In [ ]:
%cd /content
!rm -rf alpamayo_drone
!git clone https://github.com/AnuragMandke/alpamayo_drone.git
%cd /content/alpamayo_drone
!git log --oneline -1

## 3. Install the OpenVLA env deps
OpenVLA's `trust_remote_code` needs **transformers 4.40.1** (Colab ships a much newer one, so
this downgrades it). `accelerate` and `peft` are **pinned to that same era, not floored** —
Colab preinstalls versions that satisfy a `>=` floor, and a modern accelerate moves 4-bit
models with `.to()`, which transformers 4.40.1 forbids (it crashes *after* the 15 GB load).

We deliberately **do not** reinstall torch/torchvision — Colab's preinstalled build is matched
to the runtime CUDA, and replacing it from PyPI often breaks the GPU.

The version print below is the check that the downgrade actually took effect. **If it shows
anything else, restart the session** — pip cannot replace an already-imported module.

In [ ]:
!pip install -q \
  "transformers==4.40.1" "tokenizers>=0.19,<0.20" "timm==0.9.10" \
  "accelerate==0.29.3" "peft==0.11.1" "bitsandbytes>=0.43.0" \
  "Pillow>=10.0.0" "PyYAML>=6.0"

import transformers, accelerate, peft, bitsandbytes
print(f'transformers={transformers.__version__} accelerate={accelerate.__version__} '
      f'peft={peft.__version__} bitsandbytes={bitsandbytes.__version__}')
print('Expect transformers=4.40.1 accelerate=0.29.3 peft=0.11.1. If not, the old versions '
      'are still imported: Runtime -> Restart session, then re-run from cell 2.')

## 4. Bring the data over from Drive
Extracts the tarball to `/content/data_store` — **outside the repo**, deliberately — and
symlinks it in at `data/uzh_fpv` where the configs expect it.

Why outside: cell 2 does `rm -rf alpamayo_drone` to get a clean clone, and the dataset used
to live inside that directory, so **re-cloning to pick up a code fix silently deleted the
data**. Keeping it in `/content/data_store` (same trick as the HF cache in cell 5) makes the
two independent: re-clone as often as you like, the data survives and this cell becomes a
no-op. Run it once per session, after the first clone.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

TARBALL = '/content/drive/MyDrive/alpamayo/uzh_fpv.tar.gz'  # <-- edit if you used another path
STORE   = '/content/data_store'   # OUTSIDE the repo, like the HF cache below: cell 2's
                                  # `rm -rf alpamayo_drone` cannot reach it, so re-cloning
                                  # to pick up a code fix no longer destroys the dataset.

if os.path.isdir(f'{STORE}/uzh_fpv/trajectories'):
    print(f'already extracted at {STORE} — skipping (re-cloning costs no re-extract)')
else:
    os.makedirs(STORE, exist_ok=True)
    !tar xzf "{TARBALL}" -C "{STORE}"

# The configs use the relative path data/uzh_fpv, so link the store into the repo.
!mkdir -p /content/alpamayo_drone/data
!ln -sfn "{STORE}/uzh_fpv" /content/alpamayo_drone/data/uzh_fpv

trajs   = sorted(os.listdir(f'{STORE}/uzh_fpv/trajectories'))
n_poses = sum(os.path.exists(f'{STORE}/uzh_fpv/trajectories/{d}/poses.npy') for d in trajs)
print(f'trajectories: {len(trajs)}   with poses.npy: {n_poses}')
print('expect 118 / 118. If poses.npy is 0, your tarball predates commit 6789b8f —'
      ' re-convert and re-upload, or waypoint mode will fail.')

## 5. Point the HF cache at local disk
OpenVLA weights (~15 GB) download here. This is on Colab's fast ephemeral disk and is
**re-downloaded every session** — a free 15 GB Drive can't hold it, so we don't persist it.

In [ ]:
import os
os.environ['HF_HOME'] = '/content/hf_cache'

## 6. Run the probe (~50 min)
First run spends several minutes downloading the ~15 GB of weights, then trains **600 steps**
(~50 min at ~12 steps/min on a T4). The `pretrained` arm is the only one that fits a 16 GB T4
(4-bit QLoRA).

Watch the `[Train] marginal-only loss floor = ...` line printed at startup — the run is read
against it. Keep the tab active; a free-tier disconnect costs the whole 15 GB download.

In [ ]:
!python scripts/train_openvla.py --config configs/openvla_colab.yaml --init pretrained

## What success looks like
- `[Train] precision=torch.float16 (autocast=on)` and `[Train] target_mode=waypoint horizon=8`
  near the top. If it says `target_mode=velocity`, your clone is stale — re-run cell 2.
- `[Train] marginal-only loss floor = 2.563 nats` — the trainer computes this from your
  actual train split. **This is the number the whole run is read against.**
- The `loss=` line ends with `(mean of last N batches)`. If it does not, you are running
  pre-`6789b8f` code and the number is a *cumulative* mean that slides downward on its own.
- Trainable-params line shows only the LoRA adapters (33.5M / 7.57B = 0.44%).
- Ends with `[max_steps=600 reached — stopping]` and saves
  `outputs/openvla/pretrained/epoch001` (~130 MB — adapters only).

### The number that matters
**The printed marginal floor (2.563) is what a model scores knowing the action distribution
but ignoring the image entirely.** This 600-step run (~50 min, 2400 samples) exists to answer
one question: does the loss break below it?

- **Breaks below (say ~2.3 or lower)** — the model is using the camera. The pipeline works;
  book the lab GPU for the 3-arm ablation.
- **Stalls at ~2.56** — it has learned the action prior and nothing about what it sees, and
  that is worth knowing on a free T4 rather than after booking the GPU. Prime suspect is the
  attention-only LoRA (`LORA_TARGETS` = q/k/v/o_proj), where OpenVLA's official recipe uses
  `all-linear` including the MLP.

For reference, both 50-step runs landed ~0.1 nats below their floor (velocity 2.450 vs 2.576;
waypoint 2.482 vs 2.563) — i.e. essentially at the marginal. That was expected: 200 samples
buys the marginal and little else, which is exactly why this longer run exists.

Note the reported loss is diluted ~2x: it averages 8 supervised positions, but
`drone_to_openvla` writes constant zeros into roll/pitch/gripper, so 3 of the 7 action
tokens plus EOS are free to predict. Only dims `[0,1,2,5]` carry drone data — the offline
eval should score those 4 alone.

## Troubleshooting
- **CUDA OOM** — lower `batch_size` to 2 (or 1) in `configs/openvla_colab.yaml`.
- **`loss=nan`** — fp16 underflow; the GradScaler should handle it, but if it persists
  drop `optimizer.lr` and/or `batch_size`.
- **`FileNotFoundError: poses.npy`** — waypoint mode needs poses; your Drive tarball predates
  commit `6789b8f`. Re-convert and re-upload (see the header).
- **`` `.to` is not supported for `4-bit` ``, at the end of `from_pretrained`** — your
  `accelerate` is newer than `transformers==4.40.1`. Re-run the install cell (it pins
  `accelerate==0.29.3`) and **restart the session**; check the version print.
- **bf16 error inside the OpenVLA remote code** — the remote `modeling_prismatic.py` may
  hardcode a bf16 path in the vision tower. If so, that's exactly the finding this smoke
  test exists to surface — report it; the fix is a small patch to force the vision dtype.
- **Session disconnects mid-run** — free Colab idles out and this run is ~50 min; keep the
  tab active. The weights (~15 GB) re-download each session, so a disconnect is expensive.

## 7. Score the probe with the offline eval
The loss curve is the go/no-go signal; the offline eval is the *metric the ablation is compared
on*. It teacher-forces the val split and reports, for the drone dims `[vx, vy, vz, yaw_rate]`
only (not the 3 constant pad dims that dilute the loss ~2×):

- **action_token_accuracy** — top-1 over the 7 action tokens (quantization-free, scale-free)
- **action_l2 / per_dim_mae** — decoded error in physical units

It reads the run's own `drone_norm_stats.json` and now **resolves precision from the config**, so
it runs in **fp16 on a T4** (it previously hardcoded bf16, which a Turing card can't do). Scores
are saved to `outputs/openvla/pretrained/epoch001/eval_metrics.json`.

In [ ]:
!python scripts/eval_openvla.py \
  --config configs/openvla_colab.yaml --init pretrained \
  --ckpt outputs/openvla/pretrained/epoch001

## 8. The scratch control arm — pretraining, or just finetuning?
The probe proved the pipeline learns from the camera. It did **not** prove that *robot
pretraining* is why — a randomly-initialized model with the same LoRA could reach ~2.2 purely
from finetuning on the drone data. This arm settles it. Read it against the **same** printed
marginal floor (2.563):

- **scratch also breaks the floor** → pretraining wasn't necessary; the gain is finetuning.
- **scratch stalls at ~2.56 while pretrained broke it** → pretraining is doing the work. ✅

**Why this is valid on a T4 despite the precision mismatch.** Pretrained runs 4-bit QLoRA;
scratch runs full fp16 (scratch is never quantized). Quantization only *hurts*, so full-precision
scratch is the **generous** "no-pretraining" arm — if even it stalls while the 4-bit pretrained
arm broke the floor, you can't blame quantization. That's a conservative win for pretraining.

**VRAM + cost.** The scratch base is a ~14 GB fp16 backbone, so this is **borderline on a free
T4** even at `batch_size: 1` — it may OOM. If it does: Colab Pro (L4, 24 GB — raise `batch_size`
to 4 and `gradient_accumulation_steps` to 1 in `configs/openvla_colab_scratch.yaml`) or the lab
GPU. As shipped it runs `batch_size 1 × grad_accum 4 = effective batch 4`, matching the pretrained
probe (2400 samples), so the curves are directly comparable — but that's ~3–4× the wall time
(~3 h). For a faster, rougher read, set `max_steps: 300` in the config first.

In [ ]:
!python scripts/train_openvla.py --config configs/openvla_colab_scratch.yaml --init scratch

In [ ]:
!python scripts/eval_openvla.py \
  --config configs/openvla_colab_scratch.yaml --init scratch \
  --ckpt outputs/openvla/scratch/epoch001

## 9. Compare the two arms
The read is the gap between the arms on the quantization-free metric
(`action_token_accuracy`), alongside each arm's final loss vs the shared floor (2.563).
The full `scripts/ablate_openvla.py` aggregator wants all three arms (it needs `prismatic`
for a verdict), so here we just diff the two `eval_metrics.json` we have on the T4.

In [ ]:
import json, pathlib

def load(arm):
    p = pathlib.Path(f'outputs/openvla/{arm}/epoch001/eval_metrics.json')
    return json.load(open(p)) if p.exists() else None

pre, scr = load('pretrained'), load('scratch')
print(f"{'arm':<12}{'token_acc':>11}{'action_l2':>11}{'n':>7}")
for name, m in [('pretrained', pre), ('scratch', scr)]:
    if m is None:
        print(f"{name:<12}   (no eval_metrics.json yet — run its eval cell)")
    else:
        print(f"{name:<12}{m['action_token_accuracy']:>11.4f}"
              f"{m['action_l2']:>11.4f}{m['n_samples']:>7}")

if pre and scr:
    d = pre['action_token_accuracy'] - scr['action_token_accuracy']
    print(f"\ntoken-accuracy gap (pretrained - scratch): {d:+.4f}")
    print("clear positive gap  -> robot pretraining is doing the work (the thesis result)")
    print("~zero gap           -> the learning was finetuning, not transfer")